In [2]:
#5. RAG
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

# 1) Load the same embedding model you used when indexing
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# 2) Load the persisted vector store
vectorstore = Chroma(
    persist_directory="./chroma_langchain_db",
    collection_name="uia_courses",
    embedding_function=embeddings,
)

# 3) Turn it into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 4) Load your Ollama LLM
llm = ChatOllama(
    model="qwen2.5:0.5b",   # or another model you actually pulled
    temperature=0,
    base_url="http://localhost:11434"
)

# 5) Ask a question
question = "What are the learning outcomes of the deep neural network course?"

# Retrieve relevant chunks
docs = retriever.invoke(question)

# Build context
context = "\n\n".join(doc.page_content for doc in docs)

# Prompt the LLM
prompt = f"""
Answer the question using only the context below.
If the answer is not in the context, say you do not know.

Context:
{context}

Question:
{question}
"""

response = llm.invoke(prompt)

print("QUESTION:")
print(question)
print("\nRETRIEVED DOCUMENTS:")
for i, doc in enumerate(docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)

print("\nANSWER:")
print(response.content)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9559.93it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUESTION:
What are the learning outcomes of the deep neural network course?

RETRIEVED DOCUMENTS:

--- Document 1 ---
Learning outcomes:
Understand advanced concepts in deep learning, including why deep models generalize and how modern architectures are designed.
Metadata: {'ects': '7.5', 'row_id': 0, 'course_leader': 'Morten Goodwin', 'title': 'IKT469 Deep Neural Networks (Spring 2026)'}

--- Document 2 ---
Title: IKT469 Deep Neural Networks (Spring 2026)
ECTS: 7.5
Course leader: Morten Goodwin
Metadata: {'title': 'IKT469 Deep Neural Networks (Spring 2026)', 'course_leader': 'Morten Goodwin', 'ects': '7.5', 'row_id': 0}

ANSWER:
The learning outcomes of the deep neural network course are to understand advanced concepts in deep learning, including why deep models generalize and how modern architectures are designed.
